# Imports

In [ ]:
import pandas as pd
import plip_analysis as pa
from pathlib import Path
import plotly.express as px

# Load the data

In [ ]:
data_path = Path("20241120_plip_analysis")

In [ ]:
datasets = {}
for virus in ["SARS-CoV-2", "MERS-CoV"]:
    datatype_dict = {}
    for data_type in ['crystal', 'docked']:
        dataset_name = f"{virus[:4]}_{data_type}"
        datatype_dict[data_type] = {csv_path.stem: pa.PLIntReport.from_csv(csv_path) for csv_path in data_path.glob(f"{dataset_name}*.csv")}
    datasets[virus] = datatype_dict

# convert names to be easier to process

In [ ]:
import re

In [ ]:
for virus, datatype_dict in datasets.items():
    for datatype, plint_dict in datatype_dict.items():
        names = [name for name in plint_dict.keys()]
        new_plint_dict = {}
        for k, value in plint_dict.items():
            asap_id = re.search(r'(ASAP-[0-9]*)', k).group(1)
            new_plint_dict[asap_id] = value
        datatype_dict[datatype] = new_plint_dict

# Data Processing

## Get fingerprint comparison

In [ ]:
score_list = []
missing = []
for virus, datatype_dict in datasets.items():
    for level in pa.FingerprintLevel:
        for name, docked_plint_report in datatype_dict["docked"].items():
            crystal_plint_report = datatype_dict["crystal"].get(name)
            if crystal_plint_report is None:
                missing.append(name)
                continue
            print(f"Processing {name} {virus} {level}")
            print(crystal_plint_report, docked_plint_report)
            score_list.append({'ASAP_Ligand_ID': name, 'Variant': virus, **pa.InteractionScore.from_fingerprints(crystal_plint_report, docked_plint_report, level).dict()})

In [ ]:
df = pd.DataFrame.from_records(score_list)
df["ratio_of_intersection"] = df["number_of_interactions_in_intersection"] / df["number_of_interactions_in_reference"]

In [ ]:
df['ratio_of_query'] = df["number_of_interactions_in_query"] / df["number_of_interactions_in_reference"]

In [ ]:
by_interaction_type = df[df["provenance"] == pa.FingerprintLevel.ByInteractionType.value]
by_everything = df[df["provenance"] == pa.FingerprintLevel.ByEverything.value]

# Figure Making

In [ ]:
labels={'Variant': 'Viral Variant', 'tversky_index': 'PLIF Recall', 
                          'ByTotalInteractions': 
                              'Total Number of Interactions', 
                          'ByEverything': 'Atomic Level', 
                          'ByInteractionType': 'Interaction Type', 
                          'ByInteractionTypeAndAtomTypes': 'Interaction Type and Atom Type', 'ByInteractionTypeAndResidueTypeAndNumber': 'Interaction Type and Residue Type and Number'}

In [ ]:
df['Fingerprint Specificity'] = df.provenance.apply(lambda x: labels.get(x))

In [ ]:
everything_and_nothing = df[df["provenance"].isin(["ByTotalInteractions", "ByEverything"])]
filtered = df[df["provenance"].isin(["ByTotalInteractions","ByInteractionType", "ByInteractionTypeAndAtomTypes", "ByEverything"])]
interesting = df[df["provenance"].isin(["ByTotalInteractions", "ByInteractionTypeAndResidueTypeAndNumber"])]

In [ ]:
def make_fig(df, name):
    total_structures_per_variant = df.ASAP_Ligand_ID.nunique()
    fig = px.ecdf(df, 
              x="tversky_index", 
              height=600, 
              width=800, 
              # color='provenance',
              template='simple_white', 
              # line_dash="Variant",
              line_dash="Fingerprint Specificity",
              color="Variant",    
              # ecdfnorm=None,
                  ecdfnorm='probability',
                  labels=labels,
                  category_orders={'Variant': ['SARS-CoV-2', 'MERS-CoV'], 'Fingerprint Specificity': ['Interaction Type and Residue Type and Number', 'Total Number of Interactions']},
              )
    fig.update_layout(legend={'title': 'Viral Variant | Fingerprint Specificity'})
    fig.for_each_yaxis(lambda y: y.update(title=' '))
    import re
    fig.for_each_annotation(lambda x: x.update(text=re.sub(r"(\w)([A-Z])", r"\1 \2", x.text.replace('provenance=', ''))))  
    # move the legend to (0.7, 10)
    fig.update_layout(legend=dict(
        x=0.6,
        y=0,
    ))
    fig.update_layout(yaxis1=dict(title=f'Fraction of Structures (out of {total_structures_per_variant})'))
    fig.update_xaxes(title='PLIF Recall')
    # fig.update_yaxes(range=[1, total_structures_per_variant])
    fig.update_yaxes(range=[1/(total_structures_per_variant), 1])
    fig.show()
    fig.write_image(f"{name}.png")
    fig.write_image(f"{name}.svg")

In [ ]:
# make_fig(df, "20250128_cdf_fingerprint_comparison")
make_fig(everything_and_nothing, "20250128_cdf_fingerprint_comparison_extremes")
# make_fig(filtered, "20250128_cdf_fingerprint_comparison_filtered")
make_fig(interesting, "20250128_cdf_fingerprint_comparison_interesting")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Assuming your DataFrame 'df' is loaded
# Example df creation (you would use your own data)
# df = pd.read_csv('your_data.csv')

# Set your custom category orders for 'Variant' and 'Fingerprint Specificity'
variant_order = ['SARS-CoV-2', 'MERS-CoV']
fingerprint_specificity_order = ['Interaction Type and Residue Type and Number', 'Total Number of Interactions']

# Define the labels dictionary
labels = {
    'tversky_index': 'PLIF Recall',
    'Variant': 'Viral Variant',
    'Fingerprint Specificity': 'Fingerprint Specificity'
}

# Set your color palette
palette = sns.color_palette("Set2", n_colors=2)

# Create the ECDF plot
plt.figure(figsize=(10, 8))

# Plot ECDF for each combination of 'Variant' and 'Fingerprint Specificity'
for fingerprint in fingerprint_specificity_order:
    for variant in variant_order:
        subset = df[(df['Variant'] == variant) & (df['Fingerprint Specificity'] == fingerprint)]
        sns.ecdfplot(data=subset, 
                     x="tversky_index", 
                     label=f'{variant} | {fingerprint}', 
                     linewidth=2)

# Set axis labels
plt.xlabel('PLIF Recall')
plt.ylabel(f'Fraction of Structures (out of {df["ASAP_Ligand_ID"].nunique()})')

# Set plot title
plt.title('ECDF of Tversky Index by Viral Variant and Fingerprint Specificity')

# Customize the legend
plt.legend(title='Viral Variant | Fingerprint Specificity', loc='lower right', bbox_to_anchor=(0.6, 0))

# Remove y-axis title
plt.gca().yaxis.set_label_text('')

# Adjust layout
plt.tight_layout()

# Show the plot
plt.show()

